# 02 — phi^4: free vs interacting theory

Use the reference `Phi4Sampler` to draw lattice configurations and show
that (a) the free theory's per-mode variance matches the analytical
propagator, and (b) turning on the quartic coupling raises C.

In [ ]:
import math
import torch
import matplotlib.pyplot as plt
from couplnorm import coupling_from_samples
from couplnorm.samplers import Phi4Sampler, free_theory_mode_variance

torch.manual_seed(0)
N, m2 = 32, 1.0

## Free theory (lam = 0): mode variance vs analytical propagator

For the quadratic action the modes are independent Gaussians with
E[|phi_k|^2] = 1 / (m2 + 4 sin^2(pi k / N)) under the unitary FFT.

In [ ]:
free = Phi4Sampler(N=N, m2=m2, lam=0.0, step=1.0, n_therm=1500)
phi_free = free.sample(6000, seed=0)
print('acceptance:', round(free.last_acceptance, 3))

x_hat = torch.fft.rfft(phi_free, dim=-1, norm='ortho')
E = x_hat.real.pow(2) + x_hat.imag.pow(2)
empirical = E.mean(dim=0)
analytical = free_theory_mode_variance(N, m2=m2)

plt.figure(figsize=(6, 3.5))
plt.plot(analytical, 'o-', label='analytical 1/(m2+4 sin^2)')
plt.plot(empirical, 'x--', label='empirical MCMC')
plt.xlabel('mode k'); plt.ylabel('E[|phi_k|^2]'); plt.legend()
plt.title('Free-theory mode variance'); plt.tight_layout(); plt.show()

## Free vs interacting: the coupling C

In [ ]:
inter = Phi4Sampler(N=N, m2=m2, lam=1.0, step=0.7, n_therm=1500)
phi_inter = inter.sample(6000, seed=1)

C_free = coupling_from_samples(phi_free).item()
C_inter = coupling_from_samples(phi_inter).item()
print(f'C (free,  lam=0) = {C_free:.4f}')
print(f'C (inter, lam=1) = {C_inter:.4f}')

The interacting theory couples the modes through the quartic vertex, so
its spectral-energy covariance has genuine off-diagonal structure and C rises.